# MWE 47 - GPU Sparse Solver Backend Comparison

This notebook benchmarks the optional `nvmath_cudss` sparse direct backend
across the non-LBM single-phase methods exposed by `voids`:

- pore-network single-phase flow,
- TPFA finite-volume Darcy flow,
- Taylor-Hood Darcy-Darcy FEM,
- Taylor-Hood Brinkman FEM,
- stabilized USFEM Brinkman FEM.

LBM is intentionally excluded because it is not assembled as a SciPy sparse
linear system and therefore does not use `voids.linalg.solve.solve_linear_system`
or the serial FEM sparse-direct path.

In [1]:
# ruff: noqa: E402
from __future__ import annotations

import json
import os
import time
import warnings
from collections.abc import Callable
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:  # pragma: no cover - notebook convenience fallback
    display = print

benchmark_thread_count = int(
    os.environ.get("VOIDS_GPU_BENCHMARK_THREADS", str(min(32, os.cpu_count() or 1)))
)
for env_name in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
):
    os.environ[env_name] = str(benchmark_thread_count)
os.environ.setdefault("MKL_DYNAMIC", "FALSE")
os.environ.setdefault("OMP_DYNAMIC", "FALSE")

from voids.fem.singlephase import (
    FEMMapProblem,
    FEMSinglePhaseResult,
    FEniCSSolverOptions,
    solve_brinkman_taylor_hood,
    solve_brinkman_usfem,
    solve_darcy_taylor_hood,
)
from voids.fvm.singlephase import solve_tpfa
from voids.generators import generate_spanning_multiscale_blobs_matrix
from voids.image.network_extraction import extract_spanning_pore_network
from voids.image.porosity import (
    PermeabilityMap,
    permeability_map_from_porosity,
    porosity_map_from_binary,
)
from voids.paths import project_root
from voids.physics.singlephase import (
    FluidSinglePhase,
    PressureBC,
    SinglePhaseOptions,
    solve as solve_pnm_singlephase,
)

plt.ioff()

In [2]:
# User-editable inputs
output_dir = (
    project_root() / "notebooks" / "outputs" / "47_mwe_gpu_solver_backend_comparison"
)
output_dir.mkdir(parents=True, exist_ok=True)
output_prefix = (
    "gpu_sparse_solver_comparison_image300_block10_map30_extracted_pnm_cudss_all_gpus"
)

run_benchmark = True
flow_axis = "x"
pressure_inlet = 1.0
pressure_outlet = 0.0
viscosity = 1.0

# Default to a 300^3 synthetic binary image, block-averaged with 10^3 voxel
# blocks into a 30^3 continuum map. The pore network is extracted from the
# original 300^3 binary image, not from the coarse continuum map.
image_shape = (300, 300, 300)
map_shape = (30, 30, 30)
fine_voxels_per_cell = tuple(
    int(image_cells // map_cells)
    for image_cells, map_cells in zip(image_shape, map_shape)
)
if any(
    image_cells % map_cells for image_cells, map_cells in zip(image_shape, map_shape)
):
    raise ValueError("image_shape entries must be divisible by map_shape entries")
synthetic_porosity = 0.35
synthetic_blobiness_primary = 1.0
synthetic_blobiness_secondary = 3.0
synthetic_primary_weight = 0.65
synthetic_seed_start = 47_000
synthetic_max_tries = 64
cell_size = 1.0
kozeny_constant = 180.0
permeability_floor = 1.0e-6
permeability_cap = 5.0e-2
porosity_floor = 5.0e-2
gpu_device_ids: int | tuple[int, ...] | str = "all"
gpu_dtypes = ("float64", "float32")
pnm_extraction_backend = "porespy"
pnm_extraction_kwargs: dict[str, object] = {
    "flow_boundary_mode": "external_reservoir",
    "transport_geometry": "pyramids_and_cuboids",
}
pnm_conductance_model = "auto"


gpu_available_rows: list[dict[str, object]] = []
try:
    import torch

    gpu_available_rows.append(
        {
            "torch_cuda_available": bool(torch.cuda.is_available()),
            "torch_cuda_device_count": int(torch.cuda.device_count()),
            "torch_cuda_devices": "; ".join(
                torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())
            ),
            "torch_version": str(getattr(torch, "__version__", "")),
            "torch_cuda_version": str(getattr(torch.version, "cuda", "")),
        }
    )
except ImportError as exc:
    gpu_available_rows.append(
        {
            "torch_cuda_available": False,
            "torch_cuda_device_count": 0,
            "torch_cuda_devices": "",
            "torch_version": "",
            "torch_cuda_version": "",
            "torch_import_error": str(exc),
        }
    )

gpu_available = pd.DataFrame(gpu_available_rows)
display(gpu_available)

,torch_cuda_available,torch_cuda_device_count,torch_cuda_devices,torch_version,torch_cuda_version
0,True,2,NVIDIA RTX A5000; NVIDIA RTX A5000,2.10.0+cu128,12.8


In [3]:
def axis_index(axis: str) -> int:
    return {"x": 0, "y": 1, "z": 2}[axis]


def make_synthetic_inputs(
    shape: tuple[int, ...],
) -> tuple[FEMMapProblem, np.ndarray, dict[str, object]]:
    if len(shape) != 3:
        raise ValueError("This GPU benchmark expects a 3D continuum map")

    void_image, seed_used = generate_spanning_multiscale_blobs_matrix(
        shape=image_shape,
        porosity=synthetic_porosity,
        blobiness_primary=synthetic_blobiness_primary,
        blobiness_secondary=synthetic_blobiness_secondary,
        axis_index=axis_index(flow_axis),
        seed_start=synthetic_seed_start,
        max_tries=synthetic_max_tries,
        primary_weight=synthetic_primary_weight,
        periodic=True,
    )
    voxel_size = tuple(float(cell_size) / v for v in fine_voxels_per_cell)
    porosity_map = porosity_map_from_binary(
        void_image,
        block_shape=fine_voxels_per_cell,
        voxel_size=voxel_size,
        image_is_void=True,
        metadata={
            "case": "synthetic_multiscale_blobs_image300_block10_map30",
            "generator": "voids.generators.generate_spanning_multiscale_blobs_matrix",
            "seed_used": int(seed_used),
            "target_porosity": float(synthetic_porosity),
            "blobiness_primary": float(synthetic_blobiness_primary),
            "blobiness_secondary": float(synthetic_blobiness_secondary),
            "primary_weight": float(synthetic_primary_weight),
            "fine_voxels_per_cell": fine_voxels_per_cell,
        },
    )
    raw_permeability_map = permeability_map_from_porosity(
        porosity_map,
        characteristic_length=cell_size,
        kozeny_constant=kozeny_constant,
        solid_permeability=permeability_floor,
        free_flow_permeability=permeability_cap,
        max_permeability=permeability_cap,
        metadata={
            "case": "synthetic_multiscale_blobs_image300_block10_map30",
            "finite_permeability_floor_for_gpu_benchmark": permeability_floor,
            "finite_permeability_cap_for_gpu_benchmark": permeability_cap,
        },
    )
    permeability_map = PermeabilityMap(
        np.clip(raw_permeability_map.values, permeability_floor, permeability_cap),
        cell_size=raw_permeability_map.cell_size,
        origin=raw_permeability_map.origin,
        units=raw_permeability_map.units,
        metadata={
            **raw_permeability_map.metadata,
            "minimum_permeability_clamp_applied": True,
        },
    )
    metadata: dict[str, object] = {
        "synthetic_generator": "voids.generators.generate_spanning_multiscale_blobs_matrix",
        "synthetic_image_shape": image_shape,
        "synthetic_image_voxels": int(np.prod(image_shape)),
        "synthetic_seed_used": int(seed_used),
        "synthetic_target_porosity": float(synthetic_porosity),
        "synthetic_binary_porosity": float(np.mean(void_image)),
        "map_mean_porosity": float(porosity_map.mean_porosity),
        "map_min_porosity": float(np.min(porosity_map.values)),
        "map_max_porosity": float(np.max(porosity_map.values)),
        "map_mean_permeability": float(np.mean(permeability_map.values)),
        "map_min_permeability": float(np.min(permeability_map.values)),
        "map_max_permeability": float(np.max(permeability_map.values)),
        "fine_voxels_per_cell": fine_voxels_per_cell,
        "kozeny_constant": float(kozeny_constant),
        "permeability_floor": float(permeability_floor),
        "permeability_cap": float(permeability_cap),
        "porosity_floor": float(porosity_floor),
    }
    return (
        FEMMapProblem(
            permeability_map=permeability_map,
            porosity_map=porosity_map,
            viscosity=viscosity,
            porosity_floor=porosity_floor,
            permeability_floor=permeability_floor,
        ),
        void_image,
        metadata,
    )


print(
    f"Generating synthetic image {image_shape} and continuum map {map_shape}...",
    flush=True,
)
fem_problem, synthetic_void_image, synthetic_metadata = make_synthetic_inputs(map_shape)
print(
    f"Generated image porosity={float(np.mean(synthetic_void_image)):.6g}; "
    f"map mean porosity={synthetic_metadata['map_mean_porosity']:.6g}",
    flush=True,
)
tpfa_permeability_map = fem_problem.permeability_map
print(
    f"Extracting PNM network from original image {image_shape} "
    f"with backend={pnm_extraction_backend!r}...",
    flush=True,
)
pnm_extraction_start = time.perf_counter()
pnm_extraction = extract_spanning_pore_network(
    synthetic_void_image,
    voxel_size=float(cell_size) / float(fine_voxels_per_cell[0]),
    backend=pnm_extraction_backend,
    flow_axis=flow_axis,
    extraction_kwargs=dict(pnm_extraction_kwargs),
    provenance_notes={
        "notebook": "47_mwe_gpu_solver_backend_comparison",
        "source": "synthetic_void_image",
        "synthetic_image_shape": image_shape,
    },
    geometry_repairs=None,
)
pnm_extraction_seconds = time.perf_counter() - pnm_extraction_start
network = pnm_extraction.net
print(
    f"Extracted spanning PNM network: {network.Np} pores, {network.Nt} throats "
    f"(full: {pnm_extraction.net_full.Np} pores, {pnm_extraction.net_full.Nt} throats) "
    f"in {pnm_extraction_seconds:.3f} s",
    flush=True,
)
fluid = FluidSinglePhase(viscosity=viscosity)
pressure_bc = PressureBC(
    f"inlet_{flow_axis}min",
    f"outlet_{flow_axis}max",
    pin=pressure_inlet,
    pout=pressure_outlet,
)

case_metadata = pd.DataFrame(
    [
        {
            "network_pores": network.Np,
            "network_throats": network.Nt,
            "network_full_pores": pnm_extraction.net_full.Np,
            "network_full_throats": pnm_extraction.net_full.Nt,
            "network_extraction_seconds": pnm_extraction_seconds,
            "network_extraction_backend": pnm_extraction.backend,
            "network_backend_version": pnm_extraction.backend_version,
            "pnm_conductance_model": pnm_conductance_model,
            "image_shape": image_shape,
            "image_voxels": int(np.prod(image_shape)),
            "map_shape": map_shape,
            "map_cells": int(np.prod(map_shape)),
            **synthetic_metadata,
            "flow_axis": flow_axis,
            "pressure_drop": pressure_inlet - pressure_outlet,
            "gpu_device_ids": gpu_device_ids,
            "gpu_dtypes": ", ".join(gpu_dtypes),
            "benchmark_thread_count": benchmark_thread_count,
        }
    ]
)
case_metadata_path = output_dir / f"{output_prefix}_case_metadata.csv"
case_metadata.to_csv(case_metadata_path, index=False)
display(case_metadata)
print(f"Saved case metadata: {case_metadata_path}")

Generating synthetic image (300, 300, 300) and continuum map (30, 30, 30)...


Generated image porosity=0.35; map mean porosity=0.35


Extracting PNM network from original image (300, 300, 300) with backend='porespy'...


Extracted spanning PNM network: 5029 pores, 9022 throats (full: 5313 pores, 9174 throats) in 76.850 s


,network_pores,network_throats,network_full_pores,network_full_throats,network_extraction_seconds,network_extraction_backend,network_backend_version,pnm_conductance_model,image_shape,image_voxels,...,fine_voxels_per_cell,kozeny_constant,permeability_floor,permeability_cap,porosity_floor,flow_axis,pressure_drop,gpu_device_ids,gpu_dtypes,benchmark_thread_count
0,5029,9022,5313,9174,76.84955,porespy_snow2,3.0.3,auto,"(300, 300, 300)",27000000,...,"(10, 10, 10)",180.0,0.000001,0.05,0.05,x,1.0,all,"float64, float32",32


Saved case metadata: /prj/thermophase/volpatto/Work/voids/notebooks/outputs/47_mwe_gpu_solver_backend_comparison/gpu_sparse_solver_comparison_image300_block10_map30_extracted_pnm_cudss_all_gpus_case_metadata.csv


In [4]:
def cudss_parameters(dtype: str) -> dict[str, object]:
    return {"device_ids": gpu_device_ids, "dtype": dtype}


def metadata_value(metadata: dict[str, Any], key: str) -> object:
    return metadata.get(key, "")


def run_pnm_case(
    label: str, solver: str, solver_parameters: dict[str, object]
) -> dict[str, object]:
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            result = solve_pnm_singlephase(
                network,
                fluid=fluid,
                bc=pressure_bc,
                axis=flow_axis,
                options=SinglePhaseOptions(
                    conductance_model=pnm_conductance_model,
                    solver=solver,
                    solver_parameters=solver_parameters,
                ),
            )
        except Exception as exc:
            return {
                "method_family": "PNM",
                "formulation": "pore_network",
                "backend_label": label,
                "status": "failed",
                "failure": f"{type(exc).__name__}: {exc}",
                "wall_seconds": time.perf_counter() - start,
                "warning_count": len(caught),
                "warnings": "; ".join(str(item.message) for item in caught),
            }
    solver_info = dict(result.solver_info)
    return {
        "method_family": "PNM",
        "formulation": "pore_network",
        "backend_label": label,
        "status": "ok",
        "failure": "",
        "K": float(result.permeability[flow_axis]),
        "flow_rate": float(result.total_flow_rate),
        "solve_seconds": np.nan,
        "wall_seconds": time.perf_counter() - start,
        "residual_relative": float(result.residual_norm),
        "mass_balance_error": float(result.mass_balance_error),
        "solver_method": solver_info.get("method", solver),
        "solver_backend": solver_info.get("backend", ""),
        "cudss_dtype": solver_info.get("serial_sparse_nvmath_cudss_dtype", ""),
        "cudss_residual": solver_info.get(
            "serial_sparse_nvmath_cudss_relative_residual", ""
        ),
        "cudss_backend_seconds": solver_info.get(
            "serial_sparse_nvmath_cudss_backend_seconds", ""
        ),
        "cudss_device_names": solver_info.get(
            "serial_sparse_nvmath_cudss_device_names", ""
        ),
        "metadata_json": json.dumps(solver_info, sort_keys=True, default=str),
        "warning_count": len(caught),
        "warnings": "; ".join(str(item.message) for item in caught),
    }


def run_tpfa_case(
    label: str,
    solver_method: str,
    solver_parameters: dict[str, object],
) -> dict[str, object]:
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            result = solve_tpfa(
                tpfa_permeability_map,
                flow_axis=flow_axis,
                viscosity=viscosity,
                pressure_inlet=pressure_inlet,
                pressure_outlet=pressure_outlet,
                solver_method=solver_method,
                solver_parameters=solver_parameters,
            )
        except Exception as exc:
            return {
                "method_family": "TPFA",
                "formulation": "tpfa_darcy",
                "backend_label": label,
                "status": "failed",
                "failure": f"{type(exc).__name__}: {exc}",
                "wall_seconds": time.perf_counter() - start,
                "warning_count": len(caught),
                "warnings": "; ".join(str(item.message) for item in caught),
            }
    solver_info = dict(result.solver_info)
    return {
        "method_family": "TPFA",
        "formulation": "tpfa_darcy",
        "backend_label": label,
        "status": "ok",
        "failure": "",
        "K": float(result.permeability),
        "flow_rate": float(result.flow_rate),
        "solve_seconds": float(result.solve_seconds),
        "wall_seconds": time.perf_counter() - start,
        "residual_relative": float(result.residual_relative),
        "mass_balance_error": float(result.mass_balance_error),
        "matrix_nnz": int(result.matrix_nnz),
        "solver_method": result.solver_method,
        "solver_backend": solver_info.get("backend", ""),
        "cudss_dtype": solver_info.get("serial_sparse_nvmath_cudss_dtype", ""),
        "cudss_residual": solver_info.get(
            "serial_sparse_nvmath_cudss_relative_residual", ""
        ),
        "cudss_backend_seconds": solver_info.get(
            "serial_sparse_nvmath_cudss_backend_seconds", ""
        ),
        "cudss_device_names": solver_info.get(
            "serial_sparse_nvmath_cudss_device_names", ""
        ),
        "metadata_json": json.dumps(solver_info, sort_keys=True, default=str),
        "warning_count": len(caught),
        "warnings": "; ".join(str(item.message) for item in caught),
    }


FEMSolver = Callable[..., FEMSinglePhaseResult]


def run_fem_case(
    *,
    formulation: str,
    solver: FEMSolver,
    label: str,
    options: FEniCSSolverOptions,
) -> dict[str, object]:
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            solver_kwargs: dict[str, object] = {
                "flow_axis": flow_axis,
                "pressure_inlet": pressure_inlet,
                "pressure_outlet": pressure_outlet,
                "options": options,
            }
            if "brinkman" in formulation:
                solver_kwargs["nondimensional"] = True
            result = solver(fem_problem, **solver_kwargs)
        except Exception as exc:
            return {
                "method_family": "FEM",
                "formulation": formulation,
                "backend_label": label,
                "status": "failed",
                "failure": f"{type(exc).__name__}: {exc}",
                "wall_seconds": time.perf_counter() - start,
                "warning_count": len(caught),
                "warnings": "; ".join(str(item.message) for item in caught),
            }
    metadata = dict(result.metadata)
    return {
        "method_family": "FEM",
        "formulation": formulation,
        "backend_label": label,
        "status": "ok",
        "failure": "",
        "K": float(result.permeability),
        "flow_rate": float(result.flow_rate),
        "solve_seconds": float(result.solve_seconds),
        "wall_seconds": time.perf_counter() - start,
        "residual_relative": metadata_value(
            metadata, "serial_sparse_nvmath_cudss_relative_residual"
        ),
        "mass_balance_error": np.nan,
        "matrix_nnz": metadata_value(metadata, "serial_sparse_matrix_nnz"),
        "solver_method": metadata.get("linear_backend", ""),
        "solver_backend": metadata.get("serial_sparse_solver_backend", ""),
        "cudss_dtype": metadata.get("serial_sparse_nvmath_cudss_dtype", ""),
        "cudss_residual": metadata.get(
            "serial_sparse_nvmath_cudss_relative_residual", ""
        ),
        "cudss_backend_seconds": metadata.get(
            "serial_sparse_nvmath_cudss_backend_seconds", ""
        ),
        "cudss_device_names": metadata.get(
            "serial_sparse_nvmath_cudss_device_names", ""
        ),
        "metadata_json": json.dumps(metadata, sort_keys=True, default=str),
        "warning_count": len(caught),
        "warnings": "; ".join(str(item.message) for item in caught),
    }

In [5]:
backend_rows: list[dict[str, object]] = [
    {
        "method_family": "PNM",
        "formulation": "pore_network",
        "backend_label": "direct_cpu_reference",
        "solver": "direct",
        "solver_parameters": {},
    },
    {
        "method_family": "PNM",
        "formulation": "pore_network",
        "backend_label": "pardiso_cpu_reference",
        "solver": "pardiso",
        "solver_parameters": {},
    },
    {
        "method_family": "TPFA",
        "formulation": "tpfa_darcy",
        "backend_label": "direct_cpu_reference",
        "solver": "direct",
        "solver_parameters": {},
    },
    {
        "method_family": "TPFA",
        "formulation": "tpfa_darcy",
        "backend_label": "pardiso_cpu_reference",
        "solver": "pardiso",
        "solver_parameters": {},
    },
]
for dtype in gpu_dtypes:
    backend_rows.extend(
        [
            {
                "method_family": "PNM",
                "formulation": "pore_network",
                "backend_label": f"nvmath_cudss_{dtype}",
                "solver": "nvmath_cudss",
                "solver_parameters": cudss_parameters(dtype),
            },
            {
                "method_family": "TPFA",
                "formulation": "tpfa_darcy",
                "backend_label": f"nvmath_cudss_{dtype}",
                "solver": "nvmath_cudss",
                "solver_parameters": cudss_parameters(dtype),
            },
        ]
    )

fem_formulations: list[dict[str, object]] = [
    {
        "formulation": "darcy_taylor_hood_p2p1",
        "solver": solve_darcy_taylor_hood,
        "reference_label": "pardiso_cpu_reference",
        "reference_options": FEniCSSolverOptions.pardiso_direct(),
    },
    {
        "formulation": "brinkman_taylor_hood_p2p1",
        "solver": solve_brinkman_taylor_hood,
        "reference_label": "pardiso_cpu_reference",
        "reference_options": FEniCSSolverOptions.pardiso_direct(),
    },
    {
        "formulation": "brinkman_usfem_p1dg1",
        "solver": solve_brinkman_usfem,
        "reference_label": "superlu_dist_cpu_reference",
        "reference_options": FEniCSSolverOptions.direct_parallel("superlu_dist"),
    },
]
fem_gpu_backends: list[dict[str, object]] = []
for dtype in gpu_dtypes:
    fem_gpu_backends.append(
        {
            "backend_label": f"nvmath_cudss_{dtype}",
            "options": FEniCSSolverOptions.nvmath_cudss_direct(
                device_ids=gpu_device_ids,
                dtype=dtype,  # type: ignore[arg-type]
            ),
        }
    )

display(pd.DataFrame(backend_rows))
display(
    pd.DataFrame(
        [
            {
                "formulation": item["formulation"],
                "backend_label": item["reference_label"],
                "linear_backend": item["reference_options"].linear_backend,
                "role": "cpu_reference",
            }
            for item in fem_formulations
        ]
        + [
            {
                "formulation": item["formulation"],
                "backend_label": backend["backend_label"],
                "linear_backend": backend["options"].linear_backend,
                "role": "gpu_candidate",
            }
            for item in fem_formulations
            for backend in fem_gpu_backends
        ]
    )
)

,method_family,formulation,backend_label,solver,solver_parameters
0,PNM,pore_network,direct_cpu_reference,direct,{}
1,PNM,pore_network,pardiso_cpu_reference,pardiso,{}
2,TPFA,tpfa_darcy,direct_cpu_reference,direct,{}
3,TPFA,tpfa_darcy,pardiso_cpu_reference,pardiso,{}
4,PNM,pore_network,nvmath_cudss_float64,nvmath_cudss,"{'device_ids': 'all', 'dtype': 'float64'}"
5,TPFA,tpfa_darcy,nvmath_cudss_float64,nvmath_cudss,"{'device_ids': 'all', 'dtype': 'float64'}"
6,PNM,pore_network,nvmath_cudss_float32,nvmath_cudss,"{'device_ids': 'all', 'dtype': 'float32'}"
7,TPFA,tpfa_darcy,nvmath_cudss_float32,nvmath_cudss,"{'device_ids': 'all', 'dtype': 'float32'}"


,formulation,backend_label,linear_backend,role
0,darcy_taylor_hood_p2p1,pardiso_cpu_reference,pardiso,cpu_reference
1,brinkman_taylor_hood_p2p1,pardiso_cpu_reference,pardiso,cpu_reference
2,brinkman_usfem_p1dg1,superlu_dist_cpu_reference,petsc,cpu_reference
3,darcy_taylor_hood_p2p1,nvmath_cudss_float64,nvmath_cudss,gpu_candidate
4,darcy_taylor_hood_p2p1,nvmath_cudss_float32,nvmath_cudss,gpu_candidate
5,brinkman_taylor_hood_p2p1,nvmath_cudss_float64,nvmath_cudss,gpu_candidate
6,brinkman_taylor_hood_p2p1,nvmath_cudss_float32,nvmath_cudss,gpu_candidate
7,brinkman_usfem_p1dg1,nvmath_cudss_float64,nvmath_cudss,gpu_candidate
8,brinkman_usfem_p1dg1,nvmath_cudss_float32,nvmath_cudss,gpu_candidate


In [6]:
rows: list[dict[str, object]] = []
results_path = output_dir / f"{output_prefix}_results.csv"
if run_benchmark:
    for spec in backend_rows:
        print(
            f"Running row: {spec['method_family']} | {spec['formulation']} | "
            f"{spec['backend_label']}",
            flush=True,
        )
        if spec["method_family"] == "PNM":
            row = run_pnm_case(
                str(spec["backend_label"]),
                str(spec["solver"]),
                dict(spec["solver_parameters"]),
            )
        elif spec["method_family"] == "TPFA":
            row = run_tpfa_case(
                str(spec["backend_label"]),
                str(spec["solver"]),
                dict(spec["solver_parameters"]),
            )
        else:
            raise ValueError(f"Unknown method family {spec['method_family']!r}")
        rows.append(row)
        pd.DataFrame(rows).to_csv(results_path, index=False)
        print(
            f"Saved row: {row['method_family']} | {row['formulation']} | "
            f"{row['backend_label']} | {row['status']}",
            flush=True,
        )

    for formulation in fem_formulations:
        fem_backend_rows = [
            {
                "backend_label": formulation["reference_label"],
                "options": formulation["reference_options"],
            },
            *fem_gpu_backends,
        ]
        for backend in fem_backend_rows:
            print(
                f"Running row: FEM | {formulation['formulation']} | {backend['backend_label']}",
                flush=True,
            )
            row = run_fem_case(
                formulation=str(formulation["formulation"]),
                solver=formulation["solver"],  # type: ignore[arg-type]
                label=str(backend["backend_label"]),
                options=backend["options"],  # type: ignore[arg-type]
            )
            rows.append(row)
            pd.DataFrame(rows).to_csv(results_path, index=False)
            print(
                f"Saved row: {row['method_family']} | {row['formulation']} | "
                f"{row['backend_label']} | {row['status']}",
                flush=True,
            )

results = pd.DataFrame(rows)

reference_rows = (
    results[
        (results["status"] == "ok")
        & results["backend_label"].str.endswith("_cpu_reference")
    ]
    .sort_values(["method_family", "formulation", "wall_seconds"])
    .drop_duplicates(["method_family", "formulation"], keep="first")
    if not results.empty
    else pd.DataFrame()
)
reference_by_formulation = (
    reference_rows.set_index(["method_family", "formulation"])["K"].to_dict()
    if not reference_rows.empty
    else {}
)
reference_label_by_formulation = (
    reference_rows.set_index(["method_family", "formulation"])[
        "backend_label"
    ].to_dict()
    if not reference_rows.empty
    else {}
)
if not results.empty:
    results["K_reference"] = [
        reference_by_formulation.get((row.method_family, row.formulation), np.nan)
        for row in results.itertuples(index=False)
    ]
    results["K_reference_backend"] = [
        reference_label_by_formulation.get((row.method_family, row.formulation), "")
        for row in results.itertuples(index=False)
    ]
    results["K_relative_to_reference"] = np.where(
        (results["status"] == "ok") & np.isfinite(results["K_reference"]),
        np.abs(results["K"] - results["K_reference"])
        / np.maximum(np.abs(results["K_reference"]), 1.0e-300),
        np.nan,
    )

results.to_csv(results_path, index=False)
display(results)
print(f"Saved benchmark rows: {results_path}")

Running row: PNM | pore_network | direct_cpu_reference


Saved row: PNM | pore_network | direct_cpu_reference | ok


Running row: PNM | pore_network | pardiso_cpu_reference


Saved row: PNM | pore_network | pardiso_cpu_reference | ok


Running row: TPFA | tpfa_darcy | direct_cpu_reference


...

Saved row: FEM | brinkman_usfem_p1dg1 | nvmath_cudss_float64 | ok


Running row: FEM | brinkman_usfem_p1dg1 | nvmath_cudss_float32


Saved row: FEM | brinkman_usfem_p1dg1 | nvmath_cudss_float32 | ok


,method_family,formulation,backend_label,status,failure,K,flow_rate,solve_seconds,wall_seconds,residual_relative,...,cudss_residual,cudss_backend_seconds,cudss_device_names,metadata_json,warning_count,warnings,matrix_nnz,K_reference,K_reference_backend,K_relative_to_reference
0,PNM,pore_network,direct_cpu_reference,ok,,0.001355,0.040662,NaN,0.889119,0.0,...,,,,"{""backend"": ""scipy.sparse.linalg.spsolve"", ""in...",0,,NaN,0.001355,direct_cpu_reference,0.000000e+00
1,PNM,pore_network,pardiso_cpu_reference,ok,,0.001355,0.040662,NaN,1.067156,0.0,...,,,,"{""backend"": ""pypardiso"", ""info"": 0, ""method"": ...",0,,NaN,0.001355,direct_cpu_reference,1.119892e-15
2,TPFA,tpfa_darcy,direct_cpu_reference,ok,,0.001549,0.046470,1.256364,1.283211,0.0,...,,,,"{""backend"": ""scipy.sparse.linalg.spsolve"", ""in...",0,,183600,0.001549,pardiso_cpu_reference,1.399881e-16
3,TPFA,tpfa_darcy,pardiso_cpu_reference,ok,,0.001549,0.046470,0.172749,0.184056,0.0,...,,,,"{""backend"": ""pypardiso"", ""info"": 0, ""method"": ...",0,,183600,0.001549,pardiso_cpu_reference,0.000000e+00
4,PNM,pore_network,nvmath_cudss_float64,ok,,0.001355,0.040662,NaN,1.554297,0.0,...,0.0,0.106525,"(NVIDIA RTX A5000, NVIDIA RTX A5000)","{""backend"": ""nvmath.bindings.cudss"", ""info"": 0...",0,,NaN,0.001355,direct_cpu_reference,5.599459e-15
5,TPFA,tpfa_darcy,nvmath_cudss_float64,ok,,0.001549,0.046470,0.336119,0.348617,0.0,...,0.0,0.332696,"(NVIDIA RTX A5000, NVIDIA RTX A5000)","{""backend"": ""nvmath.bindings.cudss"", ""info"": 0...",0,,183600,0.001549,pardiso_cpu_reference,1.399881e-16
6,PNM,pore_network,nvmath_cudss_float32,ok,,0.001355,0.040661,NaN,0.561387,0.0,...,0.0,0.0555,"(NVIDIA RTX A5000, NVIDIA RTX A5000)","{""backend"": ""nvmath.bindings.cudss"", ""info"": 0...",0,,NaN,0.001355,direct_cpu_reference,1.917667e-06
7,TPFA,tpfa_darcy,nvmath_cudss_float32,ok,,0.001549,0.046471,0.309628,0.319129,0.0,...,0.0,0.305057,"(NVIDIA RTX A5000, NVIDIA RTX A5000)","{""backend"": ""nvmath.bindings.cudss"", ""info"": 0...",0,,183600,0.001549,pardiso_cpu_reference,2.395128e-05
8,FEM,darcy_taylor_hood_p2p1,pardiso_cpu_reference,ok,,0.002152,0.064572,115.682601,123.526530,,...,,,,"{""linear_backend"": ""pardiso"", ""mpi_rank"": 0, ""...",0,,68278966,0.002152,pardiso_cpu_reference,0.000000e+00
9,FEM,darcy_taylor_hood_p2p1,nvmath_cudss_float64,ok,,0.002152,0.064572,83.894522,90.267737,0.0,...,0.0,70.006087,"(NVIDIA RTX A5000, NVIDIA RTX A5000)","{""linear_backend"": ""nvmath_cudss"", ""mpi_rank"":...",0,,68278966,0.002152,pardiso_cpu_reference,2.014871e-16


Saved benchmark rows: /prj/thermophase/volpatto/Work/voids/notebooks/outputs/47_mwe_gpu_solver_backend_comparison/gpu_sparse_solver_comparison_image300_block10_map30_extracted_pnm_cudss_all_gpus_results.csv


In [7]:
ok = results[results["status"] == "ok"].copy() if not results.empty else pd.DataFrame()
failures = (
    results[results["status"] != "ok"].copy()
    if not results.empty
    else pd.DataFrame(
        columns=["method_family", "formulation", "backend_label", "failure"]
    )
)
display(failures)

summary = (
    ok.groupby(["method_family", "formulation", "backend_label"], dropna=False)
    .agg(
        K=("K", "first"),
        K_reference=("K_reference", "first"),
        K_reference_backend=("K_reference_backend", "first"),
        K_relative_to_reference=("K_relative_to_reference", "first"),
        solve_seconds=("solve_seconds", "first"),
        wall_seconds=("wall_seconds", "first"),
        cudss_backend_seconds=("cudss_backend_seconds", "first"),
        cudss_residual=("cudss_residual", "first"),
        matrix_nnz=("matrix_nnz", "first"),
        solver_backend=("solver_backend", "first"),
    )
    .reset_index()
    if not ok.empty
    else pd.DataFrame()
)
summary_path = output_dir / f"{output_prefix}_summary.csv"
summary.to_csv(summary_path, index=False)
display(summary)
print(f"Saved summary: {summary_path}")

,method_family,formulation,backend_label,status,failure,K,flow_rate,solve_seconds,wall_seconds,residual_relative,...,cudss_residual,cudss_backend_seconds,cudss_device_names,metadata_json,warning_count,warnings,matrix_nnz,K_reference,K_reference_backend,K_relative_to_reference


,method_family,formulation,backend_label,K,K_reference,K_reference_backend,K_relative_to_reference,solve_seconds,wall_seconds,cudss_backend_seconds,cudss_residual,matrix_nnz,solver_backend
0,FEM,brinkman_taylor_hood_p2p1,nvmath_cudss_float32,0.001475,0.001475,pardiso_cpu_reference,3.647092e-08,39.030322,45.881595,20.721222,0.0,68278966,nvmath.bindings.cudss
1,FEM,brinkman_taylor_hood_p2p1,nvmath_cudss_float64,0.001475,0.001475,pardiso_cpu_reference,2.940157e-16,90.443384,97.149243,72.135445,0.0,68278966,nvmath.bindings.cudss
2,FEM,brinkman_taylor_hood_p2p1,pardiso_cpu_reference,0.001475,0.001475,pardiso_cpu_reference,0.000000e+00,121.073150,127.963609,,,68278966,pypardiso.spsolve
3,FEM,brinkman_usfem_p1dg1,nvmath_cudss_float32,0.000508,0.000508,superlu_dist_cpu_reference,1.229742e-05,43.600002,45.099448,22.077801,0.000001,50365539,nvmath.bindings.cudss
4,FEM,brinkman_usfem_p1dg1,nvmath_cudss_float64,0.000508,0.000508,superlu_dist_cpu_reference,2.286976e-13,97.899336,99.350485,76.394306,0.0,50365539,nvmath.bindings.cudss
5,FEM,brinkman_usfem_p1dg1,superlu_dist_cpu_reference,0.000508,0.000508,superlu_dist_cpu_reference,0.000000e+00,134.153458,136.385014,,,,
6,FEM,darcy_taylor_hood_p2p1,nvmath_cudss_float32,0.002152,0.002152,pardiso_cpu_reference,2.447981e-09,36.036880,42.307890,22.359126,0.0,68278966,nvmath.bindings.cudss
7,FEM,darcy_taylor_hood_p2p1,nvmath_cudss_float64,0.002152,0.002152,pardiso_cpu_reference,2.014871e-16,83.894522,90.267737,70.006087,0.0,68278966,nvmath.bindings.cudss
8,FEM,darcy_taylor_hood_p2p1,pardiso_cpu_reference,0.002152,0.002152,pardiso_cpu_reference,0.000000e+00,115.682601,123.526530,,,68278966,pypardiso.spsolve
9,PNM,pore_network,direct_cpu_reference,0.001355,0.001355,direct_cpu_reference,0.000000e+00,NaN,0.889119,,,None,scipy.sparse.linalg.spsolve


Saved summary: /prj/thermophase/volpatto/Work/voids/notebooks/outputs/47_mwe_gpu_solver_backend_comparison/gpu_sparse_solver_comparison_image300_block10_map30_extracted_pnm_cudss_all_gpus_summary.csv


## Plots

The plots compare successful rows only. Failed rows remain in the tables and
CSVs so missing optional runtimes or unsupported configurations are visible.

In [8]:
if ok.empty:
    print("No successful rows to plot.")
else:
    plot_df = ok.copy()
    plot_df["label"] = (
        plot_df["method_family"].astype(str)
        + "\n"
        + plot_df["formulation"].astype(str)
        + "\n"
        + plot_df["backend_label"].astype(str)
    )
    x = np.arange(len(plot_df), dtype=float)
    fig, ax = plt.subplots(figsize=(14.0, 5.8), constrained_layout=True)
    ax.bar(x, plot_df["wall_seconds"], color="tab:blue", alpha=0.75)
    ax.set_yscale("log")
    ax.set_ylabel("wall time [s]")
    ax.set_title("CPU reference and nvmath/cuDSS wall time by method")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["label"], rotation=70, ha="right")
    ax.grid(axis="y", which="both", alpha=0.25)
    timing_plot_path = output_dir / f"{output_prefix}_wall_time.png"
    fig.savefig(timing_plot_path, dpi=180)
    plt.close(fig)
    print(f"Saved wall-time plot: {timing_plot_path}")

Saved wall-time plot: /prj/thermophase/volpatto/Work/voids/notebooks/outputs/47_mwe_gpu_solver_backend_comparison/gpu_sparse_solver_comparison_image300_block10_map30_extracted_pnm_cudss_all_gpus_wall_time.png


In [9]:
accuracy_df = (
    ok[ok["K_relative_to_reference"].notna()].copy() if not ok.empty else pd.DataFrame()
)
accuracy_df = accuracy_df[~accuracy_df["backend_label"].str.endswith("_cpu_reference")]
if accuracy_df.empty:
    print("No cuDSS rows with CPU references to plot.")
else:
    accuracy_df["label"] = (
        accuracy_df["method_family"].astype(str)
        + "\n"
        + accuracy_df["formulation"].astype(str)
        + "\n"
        + accuracy_df["backend_label"].astype(str)
    )
    x = np.arange(len(accuracy_df), dtype=float)
    fig, ax = plt.subplots(figsize=(12.0, 5.8), constrained_layout=True)
    ax.bar(x, accuracy_df["K_relative_to_reference"], color="tab:green", alpha=0.75)
    ax.set_yscale("log")
    ax.set_ylabel("relative K difference vs CPU reference")
    ax.set_title("nvmath/cuDSS permeability parity")
    ax.set_xticks(x)
    ax.set_xticklabels(accuracy_df["label"], rotation=70, ha="right")
    ax.grid(axis="y", which="both", alpha=0.25)
    accuracy_plot_path = output_dir / f"{output_prefix}_k_relative_error.png"
    fig.savefig(accuracy_plot_path, dpi=180)
    plt.close(fig)
    print(f"Saved K parity plot: {accuracy_plot_path}")

Saved K parity plot: /prj/thermophase/volpatto/Work/voids/notebooks/outputs/47_mwe_gpu_solver_backend_comparison/gpu_sparse_solver_comparison_image300_block10_map30_extracted_pnm_cudss_all_gpus_k_relative_error.png
